In [6]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
import seaborn as sns
import pandas as pd
from collections import Counter
import cv2
from torchvision import models
import torch.nn.functional as F
import kagglehub
import shutil
from pathlib import Path
import json

# Configuração de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

def download_and_prepare_dataset():
    """Baixa e prepara o dataset COVID-QU-Ex do Kaggle"""
    print("Baixando dataset COVID-QU-Ex...")
    
    # Download do dataset
    path = kagglehub.dataset_download("anasmohammedtahir/covidqu")
    print(f"Dataset baixado em: {path}")
    
    # Explorar estrutura completa do dataset
    print("\nExplorando estrutura completa do dataset...")
    dataset_structure = {}
    
    for root, dirs, files in os.walk(path):
        level = root.replace(str(path), '').count(os.sep)
        if level <= 4:  # Aumentar profundidade para melhor exploração
            indent = ' ' * 2 * level
            relative_path = os.path.relpath(root, path)
            print(f'{indent}{os.path.basename(root)}/ ({len(files)} arquivos)')
            
            # Guardar informação da estrutura
            dataset_structure[relative_path] = {
                'dirs': dirs.copy(),
                'files': len(files),
                'sample_files': files[:5] if files else []
            }
    
    return path, dataset_structure

class COVIDQUDataset(Dataset):
    """Dataset personalizado para COVID-QU-Ex com detecção aprimorada de estrutura"""
    
    def __init__(self, dataset_path, dataset_structure, split='train', transform=None, use_lung_masks=False):
        self.samples = []
        self.transform = transform
        self.use_lung_masks = use_lung_masks
        self.class_names = ['Normal', 'COVID-19', 'Non-COVID']
        
        # Mapeamento de labels mais flexível
        self.label_mapping = {
            'Normal': 0,
            'COVID-19': 1, 
            'Non-COVID': 2
        }
        
        self.dataset_path = Path(dataset_path)
        self.dataset_structure = dataset_structure
        self._load_samples(split)
        
        print(f"Dataset {split} carregado: {len(self.samples)} amostras")
        if len(self.samples) > 0:
            self._print_class_distribution()
        else:
            print("⚠️ AVISO: Nenhuma amostra encontrada! Verificando estrutura alternativa...")
            self._debug_dataset_structure()

    def _debug_dataset_structure(self):
        """Debug detalhado da estrutura do dataset"""
        print("\n=== DEBUG DA ESTRUTURA DO DATASET ===")
        
        # Procurar por arquivos de imagem em toda a estrutura
        image_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']:
            image_files.extend(list(self.dataset_path.rglob(ext)))
        
        print(f"Total de imagens encontradas: {len(image_files)}")
        
        if image_files:
            print("\nPrimeiras 10 imagens encontradas:")
            for i, img_path in enumerate(image_files[:10]):
                rel_path = img_path.relative_to(self.dataset_path)
                predicted_class = self._determine_class_from_path(img_path)
                print(f"  {i+1}. {rel_path} -> Classe: {predicted_class}")
            
            # Analisar distribuição por diretório
            dir_analysis = {}
            for img_path in image_files:
                parent_dir = str(img_path.parent.relative_to(self.dataset_path))
                if parent_dir not in dir_analysis:
                    dir_analysis[parent_dir] = []
                dir_analysis[parent_dir].append(img_path)
            
            print(f"\nDistribuição por diretório:")
            for dir_name, files in dir_analysis.items():
                sample_classes = [self._determine_class_from_path(f) for f in files[:5]]
                unique_classes = set([c for c in sample_classes if c])
                print(f"  {dir_name}: {len(files)} imagens, classes detectadas: {unique_classes}")

    def _find_dataset_root(self):
        """Encontra automaticamente os dados baseado na estrutura conhecida do COVID-QU-Ex"""
        print("\nProcurando estrutura do dataset COVID-QU-Ex...")
        
        # Possíveis caminhos baseados na estrutura típica do COVID-QU-Ex
        possible_paths = [
            # Estrutura padrão do COVID-QU-Ex
            self.dataset_path / "Infection Segmentation Data" / "Infection Segmentation Data",
            self.dataset_path / "Lung Segmentation Data" / "Lung Segmentation Data", 
            self.dataset_path / "Infection Segmentation Data",
            self.dataset_path / "Lung Segmentation Data",
            # Possíveis caminhos diretos
            self.dataset_path / "Train",
            self.dataset_path / "Test", 
            self.dataset_path / "Val",
            # Busca por qualquer subdiretório que contenha muitas imagens
            self.dataset_path
        ]
        
        best_path = None
        max_images = 0
        
        for path in possible_paths:
            if path.exists():
                # Contar imagens neste caminho
                image_count = 0
                for ext in ['*.png', '*.jpg', '*.jpeg']:
                    image_count += len(list(path.rglob(ext)))
                
                print(f"  {path.relative_to(self.dataset_path) if path != self.dataset_path else 'base'}: {image_count} imagens")
                
                if image_count > max_images:
                    max_images = image_count
                    best_path = path
        
        if best_path:
            print(f"Melhor caminho encontrado: {best_path.relative_to(self.dataset_path) if best_path != self.dataset_path else 'base'}")
            return best_path
        else:
            print("Usando caminho base do dataset")
            return self.dataset_path

    def _load_samples(self, split):
        """Carrega amostras com estratégia adaptativa"""
        base_data_path = self._find_dataset_root()
        
        # Estratégia 1: Tentar estrutura organizada por splits
        success = self._try_load_organized_splits(base_data_path, split)
        
        if not success:
            # Estratégia 2: Carregar tudo e dividir manualmente
            print(f"Estrutura organizada não encontrada. Carregando tudo e dividindo...")
            self._load_all_and_split_manual(base_data_path, split)

    def _try_load_organized_splits(self, base_path, split):
        """Tenta carregar dados de estrutura já organizada por splits"""
        split_variants = {
            'train': ['Train', 'train', 'training', 'Training'],
            'val': ['Val', 'val', 'validation', 'Validation', 'Valid'],
            'test': ['Test', 'test', 'testing', 'Testing']
        }
        
        found_split_path = None
        for variant in split_variants.get(split, [split]):
            potential_paths = [
                base_path / variant,
                base_path / variant / variant,  # Estruturas aninhadas
            ]
            
            for potential_path in potential_paths:
                if potential_path.exists():
                    # Verificar se tem subdiretórios de classes ou imagens diretamente
                    subdirs = [d for d in potential_path.iterdir() if d.is_dir()]
                    images = list(potential_path.glob('*.png')) + list(potential_path.glob('*.jpg'))
                    
                    if subdirs or images:
                        found_split_path = potential_path
                        break
            
            if found_split_path:
                break
        
        if found_split_path:
            print(f"Encontrado split '{split}' em: {found_split_path}")
            self._load_from_path_with_class_detection(found_split_path)
            return len(self.samples) > 0
        
        return False

    def _load_from_path_with_class_detection(self, path):
        """Carrega imagens de um caminho detectando classes automaticamente"""
        # Primeiro, tentar estrutura organizada por classes
        subdirs = [d for d in path.iterdir() if d.is_dir()]
        
        if subdirs:
            # Tentar carregar de subdiretórios (estrutura por classes)
            for subdir in subdirs:
                predicted_class = self._determine_class_from_path(subdir)
                if predicted_class:
                    label = self.label_mapping[predicted_class]
                    self._add_images_from_directory(subdir, label)
                    print(f"  Carregadas imagens de {subdir.name} como classe {predicted_class}")
        
        # Se não encontrou em subdiretórios, carregar diretamente
        if not self.samples:
            direct_images = list(path.glob('*.png')) + list(path.glob('*.jpg')) + list(path.glob('*.jpeg'))
            for img_path in direct_images:
                predicted_class = self._determine_class_from_path(img_path)
                if predicted_class:
                    label = self.label_mapping[predicted_class]
                    self.samples.append((str(img_path), label))

    def _add_images_from_directory(self, directory_path, label):
        """Adiciona imagens de um diretório"""
        image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']
        added_count = 0
        
        for ext in image_extensions:
            for img_file in directory_path.glob(ext):
                self.samples.append((str(img_file), label))
                added_count += 1
                
        # Se não encontrou no primeiro nível, tentar recursivamente
        if added_count == 0:
            for ext in image_extensions:
                for img_file in directory_path.rglob(ext):
                    self.samples.append((str(img_file), label))
                    added_count += 1
        
        print(f"    Adicionadas {added_count} imagens de {directory_path.name}")

    def _determine_class_from_path(self, img_path):
        """Determina classe baseada no caminho - versão melhorada"""
        path_str = str(img_path).lower()
        
        # Mapeamento mais robusto de padrões para classes
        class_patterns = {
            'Normal': ['normal', 'healthy', 'clear'],
            'COVID-19': ['covid', 'corona', 'sars-cov-2'],
            'Non-COVID': ['non-covid', 'noncovid', 'pneumonia', 'bacterial', 'viral', 'lung_opacity']
        }
        
        # Verificar padrões no caminho completo
        for class_name, patterns in class_patterns.items():
            for pattern in patterns:
                if pattern in path_str:
                    # Verificação especial para evitar confusão entre COVID e Non-COVID
                    if class_name == 'COVID-19' and 'non' in path_str:
                        continue
                    return class_name
        
        # Fallback: tentar baseado em números ou códigos comuns
        filename = Path(img_path).stem.lower()
        
        # Alguns datasets usam códigos numéricos ou prefixos
        if any(x in filename for x in ['0_', 'normal_', 'healthy_']):
            return 'Normal'
        elif any(x in filename for x in ['1_', 'covid_', 'corona_']):
            return 'COVID-19'  
        elif any(x in filename for x in ['2_', 'pneumonia_', 'bacterial_']):
            return 'Non-COVID'
        
        # Se não conseguiu determinar, retornar None
        return None

    def _load_all_and_split_manual(self, path, split):
        """Carrega todos os dados e faz split manual estratificado"""
        print(f"Carregando todos os dados de {path} para split manual...")
        all_samples = []
        
        # Coletar todas as imagens
        image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']
        for ext in image_extensions:
            for img_file in path.rglob(ext):
                predicted_class = self._determine_class_from_path(img_file)
                if predicted_class:
                    label = self.label_mapping[predicted_class]
                    all_samples.append((str(img_file), label))

        if not all_samples:
            print("⚠️ Nenhuma imagem com classe identificável encontrada!")
            return

        # Verificar distribuição das classes
        labels = [s[1] for s in all_samples]
        class_counts = Counter(labels)
        print(f"Distribuição total encontrada:")
        for class_idx, count in class_counts.items():
            class_name = self.class_names[class_idx]
            print(f"  {class_name}: {count} amostras")

        # Verificar se há classes suficientes para split estratificado
        min_samples_per_class = min(class_counts.values()) if class_counts else 0
        
        if min_samples_per_class < 3:
            print("⚠️ Poucas amostras por classe para split estratificado. Usando split simples.")
            # Split simples sem estratificação
            np.random.seed(42)
            np.random.shuffle(all_samples)
            
            total = len(all_samples)
            train_end = int(0.7 * total)
            val_end = int(0.85 * total)
            
            if split == 'train':
                self.samples = all_samples[:train_end]
            elif split == 'val':
                self.samples = all_samples[train_end:val_end]
            elif split == 'test':
                self.samples = all_samples[val_end:]
        else:
            # Split estratificado
            try:
                X = [s[0] for s in all_samples]  # caminhos
                y = [s[1] for s in all_samples]  # labels
                
                # Dividir em treino e temp (val + test)
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )
                
                # Dividir temp em validação e teste
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
                )
                
                if split == 'train':
                    self.samples = list(zip(X_train, y_train))
                elif split == 'val':
                    self.samples = list(zip(X_val, y_val))
                elif split == 'test':
                    self.samples = list(zip(X_test, y_test))
                    
            except ValueError as e:
                print(f"Erro no split estratificado: {e}")
                # Fallback para split simples
                self._simple_split(all_samples, split)

    def _simple_split(self, all_samples, split):
        """Split simples quando estratificado falha"""
        np.random.seed(42)
        np.random.shuffle(all_samples)
        
        total = len(all_samples)
        train_end = int(0.7 * total)
        val_end = int(0.85 * total)
        
        if split == 'train':
            self.samples = all_samples[:train_end]
        elif split == 'val':
            self.samples = all_samples[train_end:val_end]
        elif split == 'test':
            self.samples = all_samples[val_end:]

    def _print_class_distribution(self):
        """Imprime distribuição das classes"""
        if not self.samples:
            print("Nenhuma amostra para mostrar distribuição")
            return
            
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        print("Distribuição das classes:")
        total = len(self.samples)
        
        for class_idx in range(len(self.class_names)):
            count = class_counts.get(class_idx, 0)
            class_name = self.class_names[class_idx]
            percentage = (count / total) * 100 if total > 0 else 0
            print(f"  {class_name}: {count} amostras ({percentage:.1f}%)")

    def get_class_weights(self):
        """Calcula pesos das classes para balanceamento"""
        if not self.samples:
            return torch.ones(len(self.class_names))
            
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        total_samples = len(self.samples)
        
        weights = []
        for i in range(len(self.class_names)):
            if i in class_counts and class_counts[i] > 0:
                weight = total_samples / (len(self.class_names) * class_counts[i])
                weights.append(weight)
            else:
                weights.append(1.0)  # Peso padrão para classes ausentes
        
        return torch.FloatTensor(weights)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if idx >= len(self.samples):
            raise IndexError(f"Index {idx} out of range for dataset with {len(self.samples)} samples")
            
        img_path, label = self.samples[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
            
            if self.transform:
                image = self.transform(image)
            else:
                # Transformação básica se nenhuma for especificada
                image = transforms.ToTensor()(image)
                image = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(image)

            return image, label
            
        except Exception as e:
            print(f"Erro ao carregar imagem {img_path}: {e}")
            # Retornar imagem preta em caso de erro
            dummy_image = torch.zeros(3, 224, 224)
            return dummy_image, label

# Resto do código permanece igual...
def get_transforms(phase='train'):
    """Define transformações específicas para cada fase"""
    
    if phase == 'train':
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=5),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

class COVIDClassificationCNN(nn.Module):
    """CNN melhorada para classificação COVID-19"""
    
    def __init__(self, num_classes=3, pretrained=True, dropout_rate=0.3):
        super(COVIDClassificationCNN, self).__init__()
        
        # Backbone pré-treinado
        self.backbone = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # Classificador mais simples para evitar overfitting
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
        
        self._initialize_weights()

    def _initialize_weights(self):
        """Inicialização dos pesos"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

def create_balanced_dataloader(dataset, batch_size, is_train=True):
    """Cria DataLoader com balanceamento de classes"""
    if not dataset.samples:
        return None
        
    if is_train and len(dataset) > 0:
        # Calcular pesos para cada amostra
        labels = [sample[1] for sample in dataset.samples]
        class_counts = Counter(labels)
        
        # Peso inversamente proporcional à frequência da classe
        weights = []
        for label in labels:
            weights.append(1.0 / class_counts[label])
        
        # Criar sampler balanceado
        sampler = WeightedRandomSampler(
            weights=weights,
            num_samples=len(weights),
            replacement=True
        )
        
        return DataLoader(dataset, batch_size=batch_size, sampler=sampler, 
                         num_workers=2, pin_memory=True)
    else:
        return DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         num_workers=2, pin_memory=True)

class ModelTrainer:
    """Trainer melhorado com métricas por classe"""
    
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, 
                 scheduler=None, device='cpu', class_names=None):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.class_names = class_names or ['Class 0', 'Class 1', 'Class 2']
        
        # Histórico
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        self.per_class_metrics = []
        
    def train_epoch(self):
        """Treina uma época com métricas detalhadas"""
        self.model.train()
        running_loss = 0.0
        all_predictions = []
        all_labels = []
        
        for batch_idx, (images, labels) in enumerate(self.train_loader):
            images, labels = images.to(self.device), labels.to(self.device)
            
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            if (batch_idx + 1) % 50 == 0:
                print(f'  Batch {batch_idx+1}/{len(self.train_loader)}, Loss: {loss.item():.4f}')
        
        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = np.mean(np.array(all_predictions) == np.array(all_labels))
        
        # Calcular métricas por classe
        class_acc = self._calculate_per_class_accuracy(all_labels, all_predictions)
        
        return epoch_loss, epoch_acc, class_acc
    
    def validate_epoch(self):
        """Valida uma época"""
        self.model.eval()
        running_loss = 0.0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in self.val_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = np.mean(np.array(all_predictions) == np.array(all_labels))
        
        class_acc = self._calculate_per_class_accuracy(all_labels, all_predictions)
        
        return epoch_loss, epoch_acc, class_acc
    
    def _calculate_per_class_accuracy(self, labels, predictions):
        """Calcula acurácia por classe"""
        class_acc = {}
        for i, class_name in enumerate(self.class_names):
            class_mask = np.array(labels) == i
            if np.sum(class_mask) > 0:
                class_predictions = np.array(predictions)[class_mask]
                class_labels = np.array(labels)[class_mask]
                class_acc[class_name] = np.mean(class_predictions == class_labels)
            else:
                class_acc[class_name] = 0.0
        return class_acc
    
    def train(self, num_epochs, early_stopping_patience=10):
        """Treinamento com early stopping e métricas por classe"""
        best_val_acc = 0.0
        patience_counter = 0
        
        print(f"Iniciando treinamento por {num_epochs} épocas...")
        print("-" * 80)
        
        for epoch in range(num_epochs):
            print(f'Época {epoch+1}/{num_epochs}')
            
            # Treinamento
            train_loss, train_acc, train_class_acc = self.train_epoch()
            
            # Validação
            if self.val_loader is not None:
                val_loss, val_acc, val_class_acc = self.validate_epoch()
            else:
                val_loss, val_acc, val_class_acc = train_loss, train_acc, train_class_acc
                print("⚠️ Sem dados de validação, usando métricas de treino")
            
            # Scheduler
            if self.scheduler:
                old_lr = self.optimizer.param_groups[0]['lr']
                self.scheduler.step(val_loss)
                new_lr = self.optimizer.param_groups[0]['lr']
                if new_lr != old_lr:
                    print(f'Learning rate: {old_lr:.6f} -> {new_lr:.6f}')
            
            # Salvar histórico
            self.train_losses.append(train_loss)
            self.train_accuracies.append(train_acc)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)
            self.per_class_metrics.append({
                'epoch': epoch,
                'train_class_acc': train_class_acc,
                'val_class_acc': val_class_acc
            })
            
            print(f'Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}')
            print(f'Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')
            
            # Mostrar acurácia por classe
            print("Acurácia por classe (Validação):")
            for class_name, acc in val_class_acc.items():
                print(f"  {class_name}: {acc:.4f}")
            
            # Early stopping
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_acc': val_acc,
                    'val_loss': val_loss,
                    'class_acc': val_class_acc
                }, 'best_covid_model.pth')
                print(f'✓ Melhor modelo salvo! Val Acc: {val_acc:.4f}')
            else:
                patience_counter += 1
            
            if patience_counter >= early_stopping_patience:
                print(f'Early stopping após {early_stopping_patience} épocas sem melhoria')
                break
            
            print("-" * 80)
        
        return best_val_acc

def plot_training_history(self):
        """Plot do histórico de treinamento"""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        epochs = range(1, len(self.train_losses) + 1)
        
        # Loss
        ax1.plot(epochs, self.train_losses, 'b-', label='Train Loss', linewidth=2)
        ax1.plot(epochs, self.val_losses, 'r-', label='Val Loss', linewidth=2)
        ax1.set_title('Model Loss', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Accuracy
        ax2.plot(epochs, self.train_accuracies, 'b-', label='Train Acc', linewidth=2)
        ax2.plot(epochs, self.val_accuracies, 'r-', label='Val Acc', linewidth=2)
        ax2.set_title('Model Accuracy', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Acurácia por classe ao longo do tempo (validação)
        if self.per_class_metrics:
            for class_name in self.class_names:
                class_accs = [metric['val_class_acc'].get(class_name, 0) for metric in self.per_class_metrics]
                ax3.plot(epochs, class_accs, label=f'{class_name}', linewidth=2, marker='o', markersize=4)
            
            ax3.set_title('Validation Accuracy by Class', fontsize=14, fontweight='bold')
            ax3.set_xlabel('Epoch')
            ax3.set_ylabel('Class Accuracy')
            ax3.legend()
            ax3.grid(True, alpha=0.3)
        
        # Learning curve comparison
        ax4.plot(epochs, np.array(self.train_accuracies) - np.array(self.val_accuracies), 
                'g-', label='Train-Val Gap', linewidth=2)
        ax4.axhline(y=0, color='k', linestyle='--', alpha=0.5)
        ax4.set_title('Overfitting Detection', fontsize=14, fontweight='bold')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Accuracy Gap')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
        plt.show()

def evaluate_model(model, test_loader, class_names, device):
    """Avaliação completa do modelo"""
    model.eval()
    all_predictions = []
    all_labels = []
    all_probs = []
    
    print("Avaliando modelo no conjunto de teste...")
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Métricas básicas
    overall_accuracy = np.mean(np.array(all_predictions) == np.array(all_labels))
    print(f"Acurácia geral: {overall_accuracy:.4f}")
    
    # Relatório de classificação
    print("\nRelatório de Classificação:")
    print("=" * 60)
    report = classification_report(all_labels, all_predictions, 
                                 target_names=class_names, 
                                 digits=4)
    print(report)
    
    # Matriz de confusão
    cm = confusion_matrix(all_labels, all_predictions)
    
    plt.figure(figsize=(12, 5))
    
    # Matriz de confusão - valores absolutos
    plt.subplot(1, 2, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix (Absolute)', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # Matriz de confusão - normalizada
    plt.subplot(1, 2, 2)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # AUC-ROC para classificação multiclasse
    if len(class_names) > 2:
        try:
            # Converter para one-hot encoding
            y_true_onehot = np.zeros((len(all_labels), len(class_names)))
            for i, label in enumerate(all_labels):
                y_true_onehot[i, label] = 1
            
            # Calcular AUC por classe
            auc_scores = {}
            for i, class_name in enumerate(class_names):
                if np.sum(y_true_onehot[:, i]) > 0:  # Verificar se a classe existe
                    auc = roc_auc_score(y_true_onehot[:, i], np.array(all_probs)[:, i])
                    auc_scores[class_name] = auc
                    print(f"AUC-ROC {class_name}: {auc:.4f}")
            
            # AUC médio
            if auc_scores:
                mean_auc = np.mean(list(auc_scores.values()))
                print(f"AUC-ROC médio: {mean_auc:.4f}")
                
        except Exception as e:
            print(f"Erro ao calcular AUC-ROC: {e}")
    
    return {
        'accuracy': overall_accuracy,
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probs,
        'confusion_matrix': cm,
        'classification_report': report
    }

def visualize_sample_predictions(model, test_dataset, device, class_names, num_samples=12):
    """Visualiza predições em amostras do conjunto de teste"""
    model.eval()
    
    # Selecionar amostras aleatórias
    indices = np.random.choice(len(test_dataset), min(num_samples, len(test_dataset)), replace=False)
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            if i >= num_samples:
                break
                
            image, true_label = test_dataset[idx]
            image_batch = image.unsqueeze(0).to(device)
            
            output = model(image_batch)
            prob = F.softmax(output, dim=1)
            _, predicted = torch.max(output, 1)
            
            # Desnormalizar imagem para visualização
            mean = torch.tensor([0.485, 0.456, 0.406])
            std = torch.tensor([0.229, 0.224, 0.225])
            image_denorm = image.clone()
            for t, m, s in zip(image_denorm, mean, std):
                t.mul_(s).add_(m)
            image_denorm = torch.clamp(image_denorm, 0, 1)
            
            # Plot
            axes[i].imshow(image_denorm.permute(1, 2, 0))
            axes[i].axis('off')
            
            # Título com predição
            pred_class = class_names[predicted.item()]
            true_class = class_names[true_label]
            confidence = prob[0, predicted.item()].item()
            
            color = 'green' if predicted.item() == true_label else 'red'
            title = f'True: {true_class}\nPred: {pred_class}\nConf: {confidence:.3f}'
            axes[i].set_title(title, fontsize=10, color=color, fontweight='bold')
    
    # Remover eixos vazios
    for i in range(len(indices), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Sample Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('sample_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()

def save_model_info(model, results, class_names, save_path="model_info.json"):
    """Salva informações do modelo e resultados"""
    model_info = {
        'model_architecture': str(model),
        'num_parameters': sum(p.numel() for p in model.parameters()),
        'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'class_names': class_names,
        'test_accuracy': float(results['accuracy']),
        'confusion_matrix': results['confusion_matrix'].tolist(),
        'classification_report': results['classification_report']
    }
    
    with open(save_path, 'w') as f:
        json.dump(model_info, f, indent=2)
    
    print(f"Informações do modelo salvas em: {save_path}")

def main():
    """Função principal"""
    print("=" * 80)
    print("SISTEMA DE CLASSIFICAÇÃO COVID-19 COM RAIOS-X")
    print("=" * 80)
    
    try:
        # 1. Download e preparação do dataset
        dataset_path, dataset_structure = download_and_prepare_dataset()
        
        # 2. Definir transformações
        train_transform = get_transforms('train')
        val_transform = get_transforms('val')
        test_transform = get_transforms('test')
        
        # 3. Criar datasets
        print("\nCriando datasets...")
        train_dataset = COVIDQUDataset(dataset_path, dataset_structure, 'train', train_transform)
        val_dataset = COVIDQUDataset(dataset_path, dataset_structure, 'val', val_transform)
        test_dataset = COVIDQUDataset(dataset_path, dataset_structure, 'test', test_transform)
        
        # Verificar se os datasets foram carregados corretamente
        if len(train_dataset) == 0:
            print("❌ ERRO: Dataset de treino vazio! Verifique a estrutura do dataset.")
            return
        
        print(f"✓ Datasets criados com sucesso!")
        print(f"  Treino: {len(train_dataset)} amostras")
        print(f"  Validação: {len(val_dataset)} amostras")  
        print(f"  Teste: {len(test_dataset)} amostras")
        
        # 4. Criar DataLoaders
        batch_size = 16
        train_loader = create_balanced_dataloader(train_dataset, batch_size, is_train=True)
        val_loader = create_balanced_dataloader(val_dataset, batch_size, is_train=False) if len(val_dataset) > 0 else None
        test_loader = create_balanced_dataloader(test_dataset, batch_size, is_train=False) if len(test_dataset) > 0 else None
        
        if train_loader is None:
            print("❌ ERRO: Não foi possível criar o DataLoader de treino!")
            return
            
        # 5. Criar modelo
        print("\nCriando modelo...")
        model = COVIDClassificationCNN(num_classes=3, pretrained=True, dropout_rate=0.3)
        model = model.to(device)
        
        # Calcular pesos das classes para loss balanceado
        class_weights = train_dataset.get_class_weights().to(device)
        print(f"Pesos das classes: {class_weights}")
        
        # 6. Definir critério, otimizador e scheduler
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5) #verbose=True)
        
        # 7. Criar trainer
        trainer = ModelTrainer(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            class_names=train_dataset.class_names
        )
        
        # 8. Treinamento
        print(f"\nIniciando treinamento no dispositivo: {device}")
        best_val_acc = trainer.train(num_epochs=50, early_stopping_patience=10)
        
        # 9. Plot do histórico de treinamento
        trainer.plot_training_history()
        
        # 10. Carregar melhor modelo e avaliar
        if os.path.exists('best_covid_model.pth'):
            print("\nCarregando melhor modelo para avaliação...")
            checkpoint = torch.load('best_covid_model.pth', map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"Melhor modelo carregado (Val Acc: {checkpoint['val_acc']:.4f})")
        
        # 11. Avaliação no conjunto de teste
        if test_loader is not None and len(test_dataset) > 0:
            print("\n" + "="*50)
            print("AVALIAÇÃO NO CONJUNTO DE TESTE")
            print("="*50)
            
            results = evaluate_model(model, test_loader, train_dataset.class_names, device)
            
            # Visualizar predições
            visualize_sample_predictions(model, test_dataset, device, train_dataset.class_names)
            
            # Salvar informações do modelo
            save_model_info(model, results, train_dataset.class_names)
            
        else:
            print("⚠️ Conjunto de teste vazio ou não disponível")
            print("Avaliando no conjunto de validação...")
            if val_loader is not None:
                results = evaluate_model(model, val_loader, train_dataset.class_names, device)
            else:
                print("⚠️ Nem teste nem validação disponíveis. Avaliando no treino...")
                results = evaluate_model(model, train_loader, train_dataset.class_names, device)
        
        print("\n" + "="*50)
        print("TREINAMENTO CONCLUÍDO COM SUCESSO!")
        print(f"Melhor acurácia de validação: {best_val_acc:.4f}")
        print(f"Modelo salvo como: best_covid_model.pth")
        print("="*50)
        
    except Exception as e:
        print(f"❌ ERRO durante execução: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Dispositivo utilizado: cuda
SISTEMA DE CLASSIFICAÇÃO COVID-19 COM RAIOS-X
Baixando dataset COVID-QU-Ex...
Dataset baixado em: /home/jose/.cache/kagglehub/datasets/anasmohammedtahir/covidqu/versions/7

Explorando estrutura completa do dataset...
7/ (1 arquivos)
  Infection Segmentation Data/ (0 arquivos)
    Infection Segmentation Data/ (0 arquivos)
      Val/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19/ (0 arquivos)
      Test/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19/ (0 arquivos)
      Train/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19/ (0 arquivos)
  Lung Segmentation Data/ (0 arquivos)
    Lung Segmentation Data/ (0 arquivos)
      Val/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19/ (0 arquivos)
      Test/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19

Traceback (most recent call last):
  File "/tmp/ipykernel_69529/827191695.py", line 942, in main
    model = model.to(device)
            ^^^^^^^^^^^^^^^^
  File "/home/jose/anaconda3/envs/anaconda-ml-ai/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1355, in to
    return self._apply(convert)
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/jose/anaconda3/envs/anaconda-ml-ai/lib/python3.11/site-packages/torch/nn/modules/module.py", line 915, in _apply
    module._apply(fn)
  File "/home/jose/anaconda3/envs/anaconda-ml-ai/lib/python3.11/site-packages/torch/nn/modules/module.py", line 915, in _apply
    module._apply(fn)
  File "/home/jose/anaconda3/envs/anaconda-ml-ai/lib/python3.11/site-packages/torch/nn/modules/module.py", line 942, in _apply
    param_applied = fn(param)
                    ^^^^^^^^^
  File "/home/jose/anaconda3/envs/anaconda-ml-ai/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1341, in convert
    return t.to(
           ^^^^^
RuntimeEr